# Bài toán phân loại đa lớp

Chẩn đoán lỗi ổ lăn (bình thường, lỗi vòng bi trong, lỗi vòng bi ngoài, lỗi con lăn).

**Dữ liệu:** mô phỏng; khi làm bài tập thay bằng bộ CWRU Bearing Data Center.

## Tạo bảng dữ liệu

In [1]:
import numpy as np

rng = np.random.default_rng(2)
ten_lop = ["Binh thuong", "Loi vong bi trong",
           "Loi vong bi ngoai", "Loi con lan"]

# Tam cum cua 4 lop tren 6 dac trung, theo thu tu:
# RMS, dinh, he so dinh, do nhon, do lech, bien do pho 35 Hz
tam = np.array([[0.20, 0.60, 3.0, 0.0,  0.0, 0.05],   # binh thuong
                [0.80, 3.00, 3.5, 1.0,  0.5, 0.40],   # loi vong bi trong
                [0.70, 2.80, 3.2, 0.8, -0.4, 0.30],   # loi vong bi ngoai
                [0.60, 2.50, 4.0, 2.0,  0.0, 0.20]])  # loi con lan
do_lech = np.array([0.10, 0.50, 0.30, 0.50, 0.20, 0.10])

m = 150                                  # so mau moi lop
X = np.vstack([rng.normal(t, do_lech, (m, 6)) for t in tam])
X += rng.normal(0, 0.25, X.shape)        # nhieu do cua he giam sat
y = np.repeat(np.arange(len(ten_lop)), m)

print("Kich thuoc bang dac trung X:", X.shape)
print("Kich thuoc vector nhan y:   ", y.shape)
print()

ten_dac_trung = ["RMS", "Dinh", "Hs dinh", "Do nhon", "Do lech", "B.do 35Hz"]
tieu_de = "".join("%-11s" % t for t in ten_dac_trung) + "Nhan"
print(tieu_de)
print("-" * len(tieu_de))
for i in rng.choice(len(X), 10, replace=False):
    print("".join("%-11.3f" % v for v in X[i]) + str(y[i]))
print("...")

Kich thuoc bang dac trung X: (600, 6)
Kich thuoc vector nhan y:    (600,)

RMS        Dinh       Hs dinh    Do nhon    Do lech    B.do 35Hz  Nhan
----------------------------------------------------------------------
0.853      2.995      3.960      2.285      0.449      0.648      1
0.538      2.426      3.949      0.490      -0.496     -0.074     2
0.855      3.229      3.755      1.169      0.390      0.993      1
0.784      3.258      3.106      1.524      -0.763     -0.337     2
-0.094     0.526      3.121      -1.227     0.198      0.323      0
0.574      2.558      4.672      2.515      -0.497     -0.224     3
1.283      3.746      3.519      1.736      -0.283     0.463      2
1.026      2.661      3.270      0.857      0.315      0.775      1
1.043      2.984      3.920      2.002      0.154      0.607      1
1.011      2.591      3.538      1.125      0.483      0.666      1
...


## Chương trình phân loại


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=7, stratify=y)
mo_hinh = make_pipeline(StandardScaler(),
                        KNeighborsClassifier(n_neighbors=5))
mo_hinh.fit(X_tr, y_tr)
y_dd = mo_hinh.predict(X_te)

print("Ma tran nham lan:")
print(confusion_matrix(y_te, y_dd))
print()
print("Bao cao phan loai:")
print(classification_report(y_te, y_dd, target_names=ten_lop, digits=2))

Ma tran nham lan:
[[45  0  0  0]
 [ 0 41  3  1]
 [ 0  4 39  2]
 [ 0  6  1 38]]

Bao cao phan loai:
                   precision    recall  f1-score   support

      Binh thuong       1.00      1.00      1.00        45
Loi vong bi trong       0.80      0.91      0.85        45
Loi vong bi ngoai       0.91      0.87      0.89        45
      Loi con lan       0.93      0.84      0.88        45

         accuracy                           0.91       180
        macro avg       0.91      0.91      0.91       180
     weighted avg       0.91      0.91      0.91       180



## Mở rộng 1 — so sánh vài bộ phân loại

In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

rng = np.random.default_rng(1)
n = 400
X = np.vstack([rng.normal([0, 0], 1.0, (n//2, 2)),
               rng.normal([2.5, 2.0], 1.0, (n//2, 2))])
y = np.r_[np.zeros(n//2, int), np.ones(n//2, int)]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                          random_state=1, stratify=y)
mo_hinh = {"Logistic": LogisticRegression(),
           "SVM-RBF": SVC(C=1.0, kernel="rbf"),
           "K-NN": KNeighborsClassifier(n_neighbors=5)}
for ten, mo in mo_hinh.items():
    pipe = make_pipeline(StandardScaler(), mo)
    pipe.fit(X_train, y_train)
    print("---", ten)
    print(classification_report(y_test, pipe.predict(X_test), digits=2))

--- Logistic
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        60
           1       1.00      0.97      0.98        60

    accuracy                           0.98       120
   macro avg       0.98      0.98      0.98       120
weighted avg       0.98      0.98      0.98       120

--- SVM-RBF
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        60
           1       1.00      0.97      0.98        60

    accuracy                           0.98       120
   macro avg       0.98      0.98      0.98       120
weighted avg       0.98      0.98      0.98       120

--- K-NN
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        60
           1       1.00      0.93      0.97        60

    accuracy                           0.97       120
   macro avg       0.97      0.97      0.97       120
weighted avg       0.97      0.97      0

## Mở rộng 2 — tinh chỉnh siêu tham số



In [4]:
import numpy as np
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

pipe = Pipeline([("chuan_hoa", StandardScaler()),
                 ("mo_hinh", SVC())])

cv = KFold(n_splits=5, shuffle=True, random_state=0)
diem = cross_val_score(pipe, X, y, cv=cv, scoring="f1_macro")
print("F1 tung lan:", np.round(diem, 3))
print("F1 trung binh = %.3f (do lech %.3f)" % (diem.mean(), diem.std()))

luoi = {"mo_hinh__C": [0.1, 1, 10, 100],
        "mo_hinh__gamma": ["scale", 0.01, 0.1, 1]}
tim = GridSearchCV(pipe, luoi, cv=cv, scoring="f1_macro", n_jobs=-1)
tim.fit(X, y)
print("Bo tham so tot nhat:", tim.best_params_)
print("F1 tot nhat = %.3f" % tim.best_score_)

F1 tung lan: [0.974 0.95  0.987 0.962 0.95 ]
F1 trung binh = 0.965 (do lech 0.015)
Bo tham so tot nhat: {'mo_hinh__C': 1, 'mo_hinh__gamma': 0.1}
F1 tot nhat = 0.970
